# 구·시간 평균 통행속도 (`speed_hist`)

`work/speed/speed_<연도>.parquet`(도로 단위, 24시간 wide)을 구(`권역구분` = EMS `sgg_nm`과 값 일치 확인됨) × 일자 × 시간 평균으로 집계한다.

이 단계는 `speed_hist`만 만든다. **`speed_norm`(평소 속도, 학습 기간 2019~2022만 사용)과 `cong_idx`는 다음 단계**에서 만든다 — 여기서 전체 기간을 섞어 계산하면 안 된다.

In [ ]:
from pathlib import Path
import time

import pandas as pd

ROOT = Path().resolve()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent

HOUR_COLS = [f"{h:02d}시" for h in range(1, 25)]
YEARS = [p.stem.replace("speed_", "") for p in sorted((ROOT / "work" / "speed").glob("speed_*.parquet"))]
print("대상 연도:", YEARS)

In [ ]:
def build_speed_hist(year: str) -> pd.DataFrame:
    """도로 단위 speed_<year>.parquet을 구×일자×시간 평균 속도로 집계한다."""
    df = pd.read_parquet(ROOT / "work" / "speed" / f"speed_{year}.parquet")

    # 구·일자별로 먼저 wide(24시간) 상태로 평균을 낸다 (링크 단위 평균, NaN은 자동 제외).
    # 이 순서가 melt부터 하는 것보다 훨씬 가볍다 (그룹 결과가 훨씬 작음).
    grouped = (
        df.groupby(["일자", "요일", "권역구분"], as_index=False)[HOUR_COLS]
        .mean()
    )

    long = grouped.melt(
        id_vars=["일자", "요일", "권역구분"],
        value_vars=HOUR_COLS,
        var_name="hour_col",
        value_name="avg_speed_kmh",
    )
    long["hour"] = long["hour_col"].str.replace("시", "", regex=False).astype(int)
    long = long.drop(columns="hour_col").rename(columns={"권역구분": "sgg_nm"})

    return long[["일자", "요일", "sgg_nm", "hour", "avg_speed_kmh"]]

In [ ]:
frames = []
for year in YEARS:
    t0 = time.time()
    part = build_speed_hist(year)
    elapsed = time.time() - t0
    print(f"{year}: {len(part):,}행 ({elapsed:.1f}초) | 구 {part['sgg_nm'].nunique()}개 | 일자 {part['일자'].nunique()}개 | 결측 {part['avg_speed_kmh'].isna().mean():.2%}")
    frames.append(part)

speed_hist = pd.concat(frames, ignore_index=True)
print(f"\n합계: {len(speed_hist):,}행")
assert speed_hist["sgg_nm"].nunique() == 25, "서울 25개 구가 아님"
assert speed_hist["hour"].between(1, 24).all(), "시간 범위 벗어남"

In [ ]:
out_path = ROOT / "work" / "speed" / "speed_by_gu_hour.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
speed_hist.to_parquet(out_path, index=False)
print("저장:", out_path)